In [42]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , START , END 
from pydantic import BaseModel , Field 
from typing import Annotated , Literal , TypedDict
from dotenv import load_dotenv
import io 
from PIL import Image



In [43]:
load_dotenv()

True

In [44]:
model = ChatGroq(model = "llama-3.3-70b-versatile" , temperature=0.3 )

In [45]:
class sentiment_output(BaseModel):
    sentiment : Literal["Positive" , "Negative"] = Field(description="classify it into the positive and negative")
    

In [46]:
model_with_sentiment = model.with_structured_output(sentiment_output)

In [47]:
class ReviewState(TypedDict):
    review : str 
    sentiment : Literal['Positive' , 'Negative']
    diaganosis : dict
    response : str

In [48]:
def find_sentiment(state : ReviewState) -> ReviewState:
    review  = state['review']
    prompt = f"After analyzing the review classify it into the positive and negative \n {review}"
    output = model_with_sentiment.invoke(prompt).sentiment
    return {"sentiment" : output}
    

In [49]:
def cheak_sentiment(state : ReviewState ) -> Literal['positive_response' , "run_diaganosis"]:
    if state["sentiment"] == "Positive":
        return "positive_response"
    else:
        return "run_diaganosis"
    

In [50]:
def positive_response(state : ReviewState)->ReviewState:
    prompt = f"Write a warmly thank you meassage in response of this review \n {state['review']} Also ask user to leave the feedback on website"
    output = model.invoke(prompt).content
    return {"response" : output }

In [51]:
class diaganosis_output(BaseModel):
    issue_type : Literal["UX", "Performance" , "Bug" , "Support" , "Other"]= Field(description="The catigory of issue mantioned in  the review")
    tone : Literal["Angry" , "Frustrated" , "Disapponted" ,"Calm"] = Field(description="The emotional tone expressed by the user")
    urgency : Literal["Low" , "Medium" , "High"] = Field(description="How urgent or critical the issue appears to be")
    

In [52]:
diaganosis_model = model.with_structured_output(diaganosis_output)

In [53]:
def run_diaganosis(state : ReviewState) -> ReviewState:
    prompt = f"diaganose this negative review \n{state['review']} and return the issue type , bug and urgency "
    output = diaganosis_model.invoke(prompt)
    return {"diaganosis"  : output.model_dump()}

In [54]:
def  negative_response(state : ReviewState) -> ReviewState:
    diaganosis = state['diaganosis']
    prompt = f"""you are a support assistant 
    the user had a '{diaganosis['issue_type']}' issue , sounded '{diaganosis['tone']}' and marked urgency '{diaganosis['urgency']}' write the empathic 
    helpfil message
    """
    output = model.invoke(prompt).content
    return {'response' : output}

In [55]:
graph = StateGraph(ReviewState)

In [56]:
graph.add_node("sentiment" , find_sentiment)
graph.add_node("positive_response" , positive_response)
graph.add_node("run_diaganosis" , run_diaganosis)
graph.add_node("negative" , negative_response)

In [57]:
graph.add_edge(START , "sentiment")
graph.add_conditional_edges("sentiment" , cheak_sentiment)
graph.add_edge("positive_response", END)
graph.add_edge("run_diaganosis" , "negative")
graph.add_edge("negative" , END)



In [58]:
workflow = graph.compile()

In [59]:
png = workflow.get_graph().draw_mermaid_png()
img = Image.open(io.BytesIO(png))
img.show()

In [61]:
positive_test = {
    "review": "The application interface is very bad  and wastes our team hours of work!"
}
res_pos = workflow.invoke(positive_test)

print("Sentiment:", res_pos["sentiment"])
print("Response:\n", res_pos["response"])

Sentiment: Negative
Response:
 I can totally understand how frustrating it must be to experience a UX issue, especially when it's impacting your workflow. I want to start by acknowledging your frustration and apologizing for the inconvenience this has caused. I'm here to help and I'm committed to resolving this issue as quickly as possible.

I've taken note of the high urgency you've marked, and I want to assure you that I'm treating this with the utmost priority. I'll do my best to provide a prompt and effective solution to get you back on track.

Can you please provide me with more details about the UX issue you're experiencing? What specifically is happening, and what steps have you taken so far? This will help me better understand the problem and work towards a resolution.

Your satisfaction is my top priority, and I appreciate your patience and cooperation. I'm here to listen and help in any way I can. Let's work together to resolve this issue and get you the best possible experie